# 13. Merge the result files (detection metrics and speed)

Add the outputs of the training notebooks (seed 0, 1 and 2) as inputs. The notebook finds every `*_metrics.csv`, reports how many seeds each model has, and gives the mean and standard deviation of the detection metrics. Counting errors, the paired OBB-versus-HBB comparison and the density analysis come from notebook 14.

In [1]:
import os, glob
import numpy as np, pandas as pd

FOLDER = "/kaggle/input"      # searched recursively for *_metrics.csv (add the outputs of notebooks 01-12 and of the seed-1/seed-2 runs as inputs)

files = sorted(glob.glob(f"{FOLDER}/**/*_metrics.csv", recursive=True), key=os.path.getmtime)
m = pd.concat([pd.read_csv(f).assign(source_file=f) for f in files], ignore_index=True)
dups = m[m.duplicated(["dataset", "model", "box", "seed"], keep=False)]
if len(dups):
    print("WARNING: the same run appears more than once; keeping the most recently modified file for each:")
    print(dups[["dataset", "model", "box", "seed", "source_file"]].to_string())
m = m.drop_duplicates(["dataset", "model", "box", "seed"], keep="last").reset_index(drop=True)
print(len(m), "runs found")
display(m.groupby(["dataset", "model", "box"]).size().unstack("box"))
m.drop(columns="source_file").to_csv("results_all_runs.csv", index=False)

24 runs found


box              HBB  OBB
dataset model            
custom  yolo11n    2    2
        yolo26n    2    2
        yolov8n    2    2
public  yolo11n    2    2
        yolo26n    2    2
        yolov8n    2    2

In [2]:
# Detection metrics and speed: mean and standard deviation over the seeds
metrics = ["params_M_fused", "test_P", "test_R", "test_F1", "test_mAP50", "test_mAP50_95", "FPS"]
g = m.groupby(["dataset", "model", "box"])
summary = g[metrics].mean().round(4).join(g[metrics].std().round(4).add_suffix("_sd")).join(g.size().rename("n_seeds"))
display(summary)
summary.to_csv("results_detection_mean_sd.csv")
print("Saved results_all_runs.csv and results_detection_mean_sd.csv. Counting errors, the paired OBB-vs-HBB comparison and the density analysis are produced by notebook 14.")

params_M_fused  test_P  test_R  test_F1  test_mAP50  \
dataset model   box                                                        
custom  yolo11n HBB           2.597  0.9554  0.9061   0.9301      0.9438   
                OBB           2.669  0.9506  0.9153   0.9326      0.9481   
        yolo26n HBB           2.390  0.9550  0.9016   0.9276      0.9434   
                OBB           2.461  0.9541  0.9153   0.9343      0.9477   
        yolov8n HBB           3.133  0.9468  0.9053   0.9256      0.9390   
                OBB           3.204  0.9508  0.9081   0.9290      0.9432   
public  yolo11n HBB           2.597  0.9292  0.8324   0.8782      0.8949   
                OBB           2.669  0.9307  0.8296   0.8772      0.8901   
        yolo26n HBB           2.390  0.9372  0.8310   0.8808      0.8893   
                OBB           2.461  0.9390  0.8401   0.8868      0.9038   
        yolov8n HBB           3.133  0.9343  0.8330   0.8807      0.8878   
                OBB           3.204  0.9382  0.8269   0.8790      0.8958   

                     test_mAP50_95     FPS  params_M_fused_sd  test_P_sd  \
dataset model   box                                                        
custom  yolo11n HBB         0.8162   76.70                0.0     0.0069   
                OBB         0.8156   66.25                0.0     0.0025   
        yolo26n HBB         0.8132   75.40                0.0     0.0032   
                OBB         0.8214   67.95                0.0     0.0028   
        yolov8n HBB         0.8156   98.75                0.0     0.0015   
                OBB         0.8141   81.20                0.0     0.0059   
public  yolo11n HBB         0.7572   81.80                0.0     0.0036   
                OBB         0.7096   70.20                0.0     0.0079   
        yolo26n HBB         0.7563   78.00                0.0     0.0098   
                OBB         0.6986   67.00                0.0     0.0037   
        yolov8n HBB         0.7462  103.55                0.0     0.0003   
                OBB         0.7083   90.00                0.0     0.0043   

                     test_R_sd  test_F1_sd  test_mAP50_sd  test_mAP50_95_sd  \
dataset model   box                                                           
custom  yolo11n HBB     0.0018      0.0023         0.0010            0.0006   
                OBB     0.0020      0.0023         0.0025            0.0051   
        yolo26n HBB     0.0040      0.0036         0.0021            0.0023   
                OBB     0.0023      0.0025         0.0078            0.0091   
        yolov8n HBB     0.0007      0.0003         0.0014            0.0006   
                OBB     0.0078      0.0013         0.0010            0.0010   
public  yolo11n HBB     0.0033      0.0002         0.0034            0.0025   
                OBB     0.0064      0.0071         0.0014            0.0019   
        yolo26n HBB     0.0019      0.0032         0.0027            0.0034   
                OBB     0.0042      0.0007         0.0089            0.0081   
        yolov8n HBB     0.0005      0.0001         0.0075            0.0040   
                OBB     0.0059      0.0052         0.0009            0.0030   

                      FPS_sd  n_seeds  
dataset model   box                    
custom  yolo11n HBB   2.2627        2  
                OBB  10.3945        2  
        yolo26n HBB   2.9698        2  
                OBB   2.0506        2  
        yolov8n HBB   4.1719        2  
                OBB   3.1113        2  
public  yolo11n HBB   9.1924        2  
                OBB   7.3539        2  
        yolo26n HBB   0.2828        2  
                OBB   1.6971        2  
        yolov8n HBB   0.3536        2  
                OBB   0.4243        2

Saved results_all_runs.csv and results_detection_mean_sd.csv. Counting errors, the paired OBB-vs-HBB comparison and the density analysis are produced by notebook 14.
